In [1]:
!pip install sentence-transformers faiss-cpu numpy

# Step 1 — Create sample Text Documents

In [3]:
import numpy as np
import faiss
import time
from sentence_transformers import SentenceTransformer

# Step 1: Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: Sample documents
documents = [
    "The refund policy allows returns within 30 days.",
    "Employees are entitled to 20 days of paid leave.",
    "Machine learning models require training data.",
    "The company provides health insurance benefits.",
    "Customers can reset their password via email.",
    "Deep learning is a subset of artificial intelligence.",
    "Contractors do not receive paid leave benefits.",
    "The product warranty lasts for two years."
]

# Step 3: Convert documents to embeddings
doc_embeddings = model.encode(documents)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Normalize for cosine similarity
faiss.normalize_L2(doc_embeddings)

dimension = doc_embeddings.shape[1]

# Step 4: Create FAISS index
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine
index.add(doc_embeddings)

print("Number of documents in index:", index.ntotal)

/Users/gourasundarmohanty/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1534.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents in index: 8


# Query the index

In [4]:
# Step 5: Query
query = "What is the leave policy for employees?"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")
faiss.normalize_L2(query_embedding)

# Step 6: Search in index
start = time.time()

k = 3
distances, indices = index.search(query_embedding, k)

end = time.time()

print("Query:", query)
print("Top Results:")

for i, idx in enumerate(indices[0]):
    print(f"Rank {i+1}")
    print("Score:", round(distances[0][i], 3))
    print("Document:", documents[idx])

print("Search Time:", round(end - start, 6), "seconds")

Query: What is the leave policy for employees?
Top Results:
Rank 1
Score: 0.677
Document: Employees are entitled to 20 days of paid leave.
Rank 2
Score: 0.349
Document: Contractors do not receive paid leave benefits.
Rank 3
Score: 0.273
Document: The company provides health insurance benefits.
Search Time: 0.003885 seconds
